# HEATWATCH — Assignment 1 | Real NYC 311 Data
**Group 11 — Mert Sürmeli · Mehmet Batu Solar · Abdurrahman Dıraz**

**Research question:** Can we predict whether a newly opened NYC heating/hot-water 311 request will remain open for **more than 72 hours**?

**SDG:** 11 — Sustainable Cities and Communities. **Data:** [NYC 311 Open Data](https://data.cityofnewyork.us/d/erm2-nwe9). **Context:** [NYC HPD Heat and Hot Water](https://www.nyc.gov/site/hpd/services-and-information/heat-and-hot-water-information.page).

> **Interpretation warning:** Closing a service request does **not necessarily mean a heating problem has been repaired.** Our target is administrative closure delay. **72 hours is our research threshold, not an official legal deadline.**

This notebook obtains **real observations** directly from the City's official API. All numbers and graphics are computed by the code; none have been invented.

## 1. Import libraries — basic tools
**Türkçe:** `pandas` tablo analizinde, `matplotlib` grafiklerde, `scikit-learn` makine öğrenmesi modellerinde kullanılır.

In [ ]:
import sys, subprocess, importlib.util
for package, module in [('pandas','pandas'),('numpy','numpy'),('matplotlib','matplotlib'),('scikit-learn','sklearn'),('requests','requests')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
from pathlib import Path
import time, json, zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_recall_fscore_support, ConfusionMatrixDisplay
OUT = Path('heatwatch_outputs')
OUT.mkdir(exist_ok=True)
print('Libraries are ready. Output folder:', OUT.resolve())

## 2. Verify the official API (first 5 matching requests)
**Türkçe:** Belediyenin API'sine doğrudan bağlanırız ve gerçekten `HEAT/HOT WATER` şikâyeti döndürüyor mu bakarız. API çalışmazsa, bunu fark edip devam etmeyiz.

In [ ]:
URL = 'https://data.cityofnewyork.us/resource/erm2-nwe9.json'
SELECT = 'unique_key,created_date,closed_date,complaint_type,descriptor,borough,incident_zip'
WHERE = ("complaint_type = 'HEAT/HOT WATER' "
         "AND created_date >= '2024-10-01T00:00:00' "
         "AND created_date < '2025-06-01T00:00:00'")
retry = Retry(total=4, backoff_factor=2, status_forcelist=[429,500,502,503,504], allowed_methods=['GET'])
session = requests.Session()
session.mount('https://', HTTPAdapter(max_retries=retry))
response = session.get(URL, params={'$select': SELECT, '$where': WHERE,
                                    '$order': 'created_date ASC', '$limit': 5}, timeout=90)
print('HTTP:', response.status_code)
response.raise_for_status()
preview = pd.DataFrame(response.json())
print('Rows returned:', len(preview))
print('Available columns:', list(preview.columns))
if preview.empty:
    raise RuntimeError('No HEAT/HOT WATER complaints returned; check filter or dataset updates.')
display(preview.drop(columns=['unique_key', 'incident_zip'], errors='ignore'))

## 3. Download the **complete** selected heating season
**Türkçe:** NYC'nin 22 milyondan fazla satırlık tüm verisini indirmiyoruz. Yalnızca 2024 Ekim–2025 Mayıs arası ısınma/sıcak su kayıtlarını sayfa sayfa alıyoruz. İndirme biterse sonuçların güvenilirliğini kontrol edebiliriz.

Aynı kaydı iki kez almamak için `unique_key` ile yinelenenleri ayıklıyoruz. `MAX_ROWS` güvenlik sınırıdır; bir sezonda bu sınır dolarsa **uyarı verip analizi durduruyoruz**, eksik veriyle gizlice ilerlemiyoruz.

In [ ]:
PAGE_SIZE = 10000
MAX_ROWS = 300000
records = []
offset = 0
while True:
    params = {'$select': SELECT, '$where': WHERE,
              '$order': 'created_date ASC, unique_key ASC',
              '$limit': PAGE_SIZE, '$offset': offset}
    r = session.get(URL, params=params, timeout=120)
    r.raise_for_status()
    chunk = r.json()
    if not isinstance(chunk, list):
        raise RuntimeError('API response is not a list: ' + str(chunk)[:400])
    records.extend(chunk)
    print(f'Fetched {len(records):,} rows...', flush=True)
    if len(chunk) < PAGE_SIZE: break
    offset += PAGE_SIZE
    if len(records) >= MAX_ROWS:
        raise RuntimeError('Safety limit reached; the season was not downloaded in full. Raise MAX_ROWS and rerun.')
    time.sleep(0.25)
raw = pd.DataFrame.from_records(records)
if raw.empty: raise RuntimeError('No data downloaded.')
print('Total raw records:', len(raw), '| Columns:', list(raw.columns))
raw.to_csv(OUT/'nyc_heat_2024_2025_raw.csv', index=False)

## 4. Clean data and create the target variable
**Türkçe:** İki tarih arasındaki fark şikâyetin sistemde kaç saat açık kaldığını gösterir. 72 saati aşan kapanışlara `1`, aşmayanlara `0` veririz. Kapanma tarihi boşsa gerçek süresini bilemediğimizden modeli eğitirken kullanmayız; kaç tanesini çıkardığımızı raporlarız.

`closed_date`, `status`, `resolution_description`, `due_date` gibi sonuç ortaya çıktıktan sonra bilinen bilgileri modele **girdi olarak vermiyoruz** (data leakage).

In [ ]:
df = raw.copy()
df = df.drop_duplicates(subset=['unique_key']).copy()
for col in ['created_date','closed_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
missing_created = df['created_date'].isna().sum()
missing_closed = df['closed_date'].isna().sum()
df['duration_hours'] = (df['closed_date']-df['created_date']).dt.total_seconds()/3600
negative_duration = (df['duration_hours'] < 0).sum()
usable = df[df['created_date'].notna() & df['duration_hours'].notna() & (df['duration_hours'] >= 0)].copy()
usable['delayed_72h'] = (usable['duration_hours'] > 72).astype(int)
usable['month'] = usable['created_date'].dt.month.astype(str)
usable['weekday'] = usable['created_date'].dt.dayofweek.astype(str)   # 0=Monday
usable['hour'] = usable['created_date'].dt.hour.astype(str)
for c in ['descriptor','borough']:
    usable[c] = usable[c].fillna('UNKNOWN').astype(str).str.strip().replace('', 'UNKNOWN')
print('Total downloaded:', len(df))
print('Missing opening dates:', missing_created)
print('Missing closure dates:', missing_closed)
print('Negative closure durations excluded:', negative_duration)
print('Usable CLOSED records:', len(usable))
print('Delayed >72 hours:', int(usable['delayed_72h'].sum()))
print('Delayed share:', f"{usable['delayed_72h'].mean()*100:.2f}%")
print('Median administrative closure hours:', round(usable['duration_hours'].median(),1))
print('NOTE: These are closure times, not verified repair times.')
usable[['created_date','duration_hours','delayed_72h','descriptor','borough','month','weekday','hour']].to_csv(OUT/'cleaned_eda.csv',index=False)

## 5. EDA — Exploratory Data Analysis
**Türkçe:** EDA, veri setini tanımak için yapılan grafik ve temel istatistik çalışmasıdır. Sırayla aylık şikâyet sayısını, kapanma süresi dağılımını, ilçelere göre gecikme paylarını ve şikâyet alt türlerini inceliyoruz.

In [ ]:
def saveplot(name):
    plt.tight_layout()
    plt.savefig(OUT/name, dpi=160, bbox_inches='tight')
    plt.show()

# Chart 1 — record counts by month: include OPEN cases, too
raw_month = df.dropna(subset=['created_date']).copy()
raw_month['year_month'] = raw_month['created_date'].dt.to_period('M').astype(str)
counts = raw_month['year_month'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9,4))
counts.plot(kind='bar',ax=ax, color='#285c75')
ax.set(title='Heating/Hot Water 311 requests by month', xlabel='Month', ylabel='Number of requests')
saveplot('01_monthly_counts.png')

# Chart 2 — duration: trim VISUALISATION at p99 only; do not delete model data
x = usable['duration_hours']
cap = x.quantile(0.99)
fig, ax = plt.subplots(figsize=(8,4))
ax.hist(x.clip(upper=cap), bins=35, color='#52859a', edgecolor='white')
ax.axvline(72, color='#bd613e', linestyle='--',label='72h research threshold')
ax.set(title='Administrative closure time distribution (99th percentile clipped for display)', xlabel='Hours to closure',ylabel='Number of requests')
ax.legend()
saveplot('02_closure_duration.png')

# Chart 3 — borough delay rate
by_borough = usable.groupby('borough')['delayed_72h'].agg(['mean','size']).query('size >= 100').sort_values('mean')
fig, ax = plt.subplots(figsize=(8,4))
(by_borough['mean']*100).plot(kind='barh',ax=ax,color='#467e87')
ax.set(title='Share closed after 72h, by borough (min. 100 records)', xlabel='Percent (%)', ylabel='Borough')
saveplot('03_borough_delay_rate.png')

# Chart 4 — top complaint details, including UNKNOWN
by_desc = usable['descriptor'].value_counts().head(10).sort_values()
fig, ax = plt.subplots(figsize=(8,4))
by_desc.plot(kind='barh',ax=ax,color='#597b99')
ax.set(title='Most frequent complaint details',xlabel='Number of closed requests', ylabel='Problem detail')
saveplot('04_complaint_details.png')

print('Borough sample counts and delayed ratios:')
display(by_borough.assign(pct_late=(by_borough['mean']*100).round(1)))

## 6. Model experiment — compare three approaches
**Türkçe:** Başvuruların tarihine göre ayırıyoruz. **Ekim–Şubat eğitim**, **Mart–Mayıs test**. Model hiç görmediği sonraki ayları tahmin etmeye çalışıyor. Basit referans model (Dummy), Logistic Regression ve Random Forest sonuçlarını karşılaştırıyoruz.

**ÇOK ÖNEMLİ:** Modele `closed_date` veya `duration_hours` girdi vermiyoruz; bunlar geleceği bilen değişkenler!

In [ ]:
cutoff = pd.Timestamp('2025-03-01')
train = usable.loc[usable['created_date'] < cutoff].copy()
test = usable.loc[usable['created_date'] >= cutoff].copy()
features = ['descriptor', 'borough', 'month', 'weekday', 'hour']
X_train, y_train = train[features], train['delayed_72h']
X_test, y_test = test[features], test['delayed_72h']
print('Train:', len(train), 'Test:', len(test))
print('Train delay share:',round(y_train.mean(),3),'Test delay share:',round(y_test.mean(),3))
if len(train)<500 or len(test)<100 or y_train.nunique()<2 or y_test.nunique()<2:
    raise ValueError('Too few samples/classes for a meaningful test. Check API download and date coverage.')
transformer=ColumnTransformer([('categorical',OneHotEncoder(handle_unknown='ignore'),features)])
models={
    'Prior-only baseline': DummyClassifier(strategy='prior'),
    'Logistic regression': Pipeline([('features',transformer),('model', LogisticRegression(max_iter=1200))]),
    'Random forest': Pipeline([('features',transformer),('model',RandomForestClassifier(n_estimators=90,max_depth=14,min_samples_leaf=25,random_state=42,n_jobs=-1))])
}
score_rows=[]
predictions={}
for name, model in models.items():
    model.fit(X_train,y_train)
    probs=model.predict_proba(X_test)[:,list(model.classes_).index(1)]
    binary=(probs>=0.5).astype(int)
    precision, recall, f1, _=precision_recall_fscore_support(y_test,binary,average='binary',zero_division=0)
    row={'model':name,'ROC_AUC':roc_auc_score(y_test,probs),
         'PR_AUC':average_precision_score(y_test,probs),'Brier':brier_score_loss(y_test,probs),
         'precision_at_0.5':precision,'recall_at_0.5':recall,'f1_at_0.5':f1}
    score_rows.append(row)
    predictions[name] = binary
    print(name, '| ROC AUC:', round(row['ROC_AUC'],3), '| PR AUC:',round(row['PR_AUC'],3))
scores=pd.DataFrame(score_rows).sort_values('PR_AUC',ascending=False)
scores.to_csv(OUT/'model_results.csv',index=False)
display(scores.round(3))
fig, ax=plt.subplots(figsize=(9,4))
scores.set_index('model')[['ROC_AUC','PR_AUC']].plot(kind='bar',ax=ax)
ax.set(ylim=(0,1),title='Model comparison on later months',ylabel='Score (higher is better)',xlabel='')
saveplot('05_model_comparison.png')

## 7. Export results + automatically generated factual summary
**Türkçe:** Gerçek sonuçları CSV dosyalarına kaydederiz. Bu bilgiler Concept Note ve slaytlarda kullanılacak. `heatwatch_results.zip` dosyasını bilgisayarına indirip bana buraya yükleyebilirsin.

In [ ]:
summary = {
    'official_dataset': 'NYC 311 Service Requests from 2020 to Present, erm2-nwe9',
    'source': 'https://data.cityofnewyork.us/d/erm2-nwe9',
    'sample_dates': '2024-10-01 to 2025-05-31',
    'total_requests': int(len(df)),
    'missing_closed_date': int(missing_closed),
    'excluded_negative_duration': int(negative_duration),
    'usable_closed_requests': int(len(usable)),
    'delay_threshold_hours': 72,
    'delayed_closed_requests': int(usable['delayed_72h'].sum()),
    'delayed_percentage': round(float(usable['delayed_72h'].mean()*100),2),
    'median_closure_hours': round(float(usable['duration_hours'].median()),2),
    'training_records': int(len(train)),
    'testing_records': int(len(test)),
    'model_results': scores.round(4).to_dict(orient='records'),
    'important_caveat': 'Closure date is an administrative endpoint, not evidence that heat was repaired.'
}
(OUT/'factual_summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
for k,v in summary.items():
    if k!='model_results': print(f'{k}: {v}')
archive='HeatWatch_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.glob('*')):
        if f.name != 'nyc_heat_2024_2025_raw.csv':
            z.write(f,arcname=f.name)
print('READY:',archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Outside Google Colab: download manually from the file browser.')

## 8. Sunumda anlatacağımız kısa hikâye (doldurulacak; sayı uydurma!)
**Problem:** Some households report missing heat or hot water through NYC 311. Our question is not whether the problem is repaired, but whether the **administrative case** closes after 72 hours.

**Data:** We use official NYC 311 records from the 2024–25 heating season. We inspect missing dates and abnormal durations before analysis.

**EDA:** Use the **actual** monthly count, time-to-closure histogram, borough comparison and complaint detail charts generated above. Note that a borough difference **does not prove** a borough caused delays.

**AI:** We train with Oct–Feb data and test on Mar–May. We compare a prior-only reference with logistic regression and random forest. Use the real `model_results.csv` values.

**Limits:** Status closed is not proof of repair; open cases and missing closure dates must be disclosed; New York trends are not directly transferable to İstanbul; no action should be automated based solely on prediction.

**Next assignment:** Build a simple user interface that asks for complaint type, borough and day/time, then shows a probability with a clear caveat.

**Never present this as an operational municipal risk system:** this is a classroom prototype.